In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DateType
from datetime import datetime

SOURCE_PATH     = "/Volumes/bfsi/landing/customers/"
TABLE_NAME      = "bfsi.bronze_layer.customers"
SCHEMA          = "bronze_layer"
CHECKPOINT_PATH = f"/Volumes/bfsi/landing/customers/_checkpoint/{SCHEMA}_customers"
BATCH_ID        = dbutils.widgets.get("batch_id") if "batch_id" in [w.name for w in dbutils.widgets.getAll()] else f"BATCH_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

In [0]:
customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("full_name", StringType(), True),
    StructField("pan_number", StringType(), True),
    StructField("aadhaar_last4", StringType(), True),
    StructField("mobile_number", StringType(), True),
    StructField("email", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("address", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("pincode", StringType(), True),
    StructField("account_type", StringType(), True),
    StructField("risk_rating", StringType(), True),        # LOW/MEDIUM/HIGH
    StructField("kyc_status", StringType(), True)          # VERIFIED/PENDING
])

# Auto Loader: incremental ingestion using cloudFiles
df_customers_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("header", "true")
    .schema(customer_schema)
    .load(SOURCE_PATH)
    .withColumn("_source_system", F.lit("CORE_BANKING"))
    .withColumn("_load_ts", F.current_timestamp())
    .withColumn("_file_name", F.col("_metadata.file_path"))
    .withColumn("_batch_id", F.lit(BATCH_ID))
    .drop("_metadata")
)

# Write stream to Delta table in bronze_layer
query = (
    df_customers_stream.writeStream
    .format("delta")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(TABLE_NAME)
)

query.awaitTermination()
print(f"Auto Loader stream completed for {TABLE_NAME}")

In [0]:
# Verify data in bronze Delta table
df_verify = spark.table(TABLE_NAME)
print(f"Total customers ingested: {df_verify.count()}")
display(df_verify.count())